# 预补偿验证：从已保存零臂继续

本入口修正写端 Tensor/PIL 预处理的内存布局差异，保留原有一致性门槛、读者、码格、预算、四个来源和两个重建种子。质量仍只报告。真实 L4 上的修复效果尚未验证。

另包含 PyTorch 2.14 对历史 checkpoint 的兼容读取修复，启动输出与报错完整存入 ZIP；不重新设计方法或放宽前向门槛。模型执行目录仍为 continuation_layout_v2，metadata 下的 launch 编号只区分保存的启动诊断。

复用 Drive 中 precompensation-20261108/batch 的 4 个未标记捕获点、12 份零臂 RGB/特征。不重新生成基准图或重建零臂；不会覆盖原失败结果。由于旧版本未保存 RGB 梯度，本次需补算 8 条 RGB VJP 和 4 次代理重建前向，然后计算尚未执行的潜变量 VJP 与写入。

全部可写时新增最多 16 次正式生成、32 次正式重建、48 次公开读出，另有 4 次代理重建（新增完整重建前向共 36 次）、4 次源末段前向及 8 次潜变量 VJP。旧调用单独保留并累计。

在 Colab 选择 GPU（上次使用 L4），设置 HF_TOKEN 后 Run all。首格仅挂载 Drive。已完成或已中断的本次续跑仍只读，不自动再次尝试。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import base64
import io
import json
import os
from pathlib import Path, PurePosixPath
import re
import subprocess
import sys
import tempfile
import zipfile
from importlib.metadata import PackageNotFoundError, version

RUN_ROOT = Path("/content/drive/MyDrive/CEG-WM/channel-precompensation-v1/precompensation-20261108")
CONFIG_RELATIVE = Path("configs/channel_precompensation_v1_colab.json")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, data):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def unpack_source(payload, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if relative.is_absolute() or ".." in relative.parts or "\\" in info.filename:
                raise ValueError("Source archive contains an unsafe path")
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Source archive must not contain symlinks")
            target = destination.joinpath(*relative.parts)
            if not target.resolve().is_relative_to(destination):
                raise ValueError("Source archive escapes its destination")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(info))
    return destination


def terminal_counts(summary):
    stages = summary.get("stages", {})
    return bool(stages and not any(value == "started" for value in stages.values())
                and stages.get("cleanup") in {"complete", "failed", "no_provider"})


def ensure_runtime(checkout, run_root):
    import torch
    parsed = re.match(r"(\d+)\.(\d+)", torch.__version__)
    metadata = {"torch": torch.__version__, "cuda_available": bool(torch.cuda.is_available())}
    if metadata["cuda_available"]:
        properties = torch.cuda.get_device_properties(0)
        metadata.update(gpu_name=properties.name, gpu_total_memory_bytes=properties.total_memory)
    write_json(run_root / "environment.json", metadata)
    if parsed is None or tuple(map(int, parsed.groups())) < (2, 6) or not metadata["cuda_available"]:
        raise RuntimeError("Use a Colab GPU runtime with CUDA PyTorch >=2.6. PyTorch/CUDA will not be reinstalled.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                    str(checkout / "requirements-reconstruction-channel-colab.txt")], check=True)
    for package in ("diffusers", "transformers", "accelerate", "sentencepiece", "protobuf", "numpy", "Pillow"):
        try:
            metadata[package] = version(package)
        except PackageNotFoundError:
            metadata[package] = None
    write_json(run_root / "environment.json", metadata)
    print(metadata)
    from google.colab import userdata
    token = os.environ.get("HF_TOKEN") or userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Add HF_TOKEN to Colab Secrets before a new run.")
    environment = dict(os.environ)
    environment["HF_TOKEN"] = token
    environment["PYTHONPATH"] = str(checkout / "src") + os.pathsep + str(checkout)
    return environment


def run_or_show(config, checkout, run_root, payload):
    run_root = Path(run_root)
    run_root.mkdir(parents=True, exist_ok=True)
    configuration = run_root / "config.json"
    if configuration.exists() and read_json(configuration) != config:
        raise RuntimeError("Existing run configuration differs; evidence retained and no computation started.")
    if not configuration.exists():
        write_json(configuration, config)
    output = run_root / "batch"
    if output.exists():
        print({"action": "read_only_existing", "output": str(output),
               "note": "Completed, failed and interrupted runs are all retained without automatic retry."})
        return read_json(output / "summary.json")
    # With no physical batch yet, this is the snapshot for the forthcoming
    # attempt, including when environment setup subsequently fails.
    (run_root / "source_snapshot.zip").write_bytes(payload)
    environment = ensure_runtime(checkout, run_root)
    command = [sys.executable, "-m", "experiments.run_channel_precompensation_v1",
               "--config", str(checkout / CONFIG_RELATIVE), "--execute", "--output", str(output)]
    try:
        completed = subprocess.run(command, cwd=checkout, env=environment, check=False)
    except BaseException as error:
        write_json(run_root / "launch_error.json", {"error": type(error).__name__ + ": " + str(error)})
        raise
    if not (output / "summary.json").exists():
        raise RuntimeError("Runner produced no readable summary; inspect saved errors and do not rerun automatically.")
    summary = read_json(output / "summary.json")
    print({"runner_exit": completed.returncode, "method_summaries": summary.get("method_summaries", {}),
           "note": "An unreachable write remains in the denominator; a nonzero exit is not permission to retry."})
    return summary


def package_results(config, run_root):
    run_root = Path(run_root)
    run_root.mkdir(parents=True, exist_ok=True)
    summary_path = run_root / "batch/summary.json"
    try:
        summary = read_json(summary_path) if summary_path.exists() else {}
    except (OSError, ValueError) as error:
        summary = {"read_error": str(error)}
    rows = {row["source_id"]: row for row in summary.get("sources", [])}
    terminal = terminal_counts(summary)
    result = {"scope": "four_source_one_shot_falsification_only", "config": config,
              "counts_terminal": terminal,
              "actual_calls": summary.get("actual_calls") if terminal else None,
              "last_recorded_calls": summary.get("actual_calls", {}),
              "method_summaries": summary.get("method_summaries", {}),
              "sources": [rows.get(row["source_id"], {"source_id": row["source_id"], "status": "not_observed"})
                          for row in config["sources"]],
              "claim": "No scientific PASS, watermark-presence, attribution, FPR or general robustness conclusion."}
    write_json(run_root / "measurement_summary.json", result)
    destination = run_root.parent / (run_root.name + "-results.zip")
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run_root.rglob("*")):
            if path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(run_root.resolve()):
                archive.write(path, Path(run_root.name) / path.relative_to(run_root))
    return result, destination

RESUME_BATCH = RUN_ROOT / "batch"
CONTINUATION_NAME = "continuation_layout_v2"
CONTINUATION_LAUNCH_METADATA = None


def new_continuation_metadata(run_root):
    root = Path(run_root) / (CONTINUATION_NAME + "_metadata")
    root.mkdir(parents=True, exist_ok=True)
    index = 1
    while True:
        destination = root / ("launch_" + str(index).zfill(4))
        try:
            destination.mkdir()
            return destination
        except FileExistsError:
            index += 1


def run_or_show_continuation(checkout, run_root, payload):
    global CONTINUATION_LAUNCH_METADATA
    CONTINUATION_LAUNCH_METADATA = None
    run_root = Path(run_root)
    original = run_root / "batch"
    if not (original / "config.json").is_file() or not (original / "summary.json").is_file():
        raise RuntimeError("The original saved batch is required; no zero images will be regenerated.")
    output = run_root / CONTINUATION_NAME
    if output.exists():
        print({"action": "read_only_existing_continuation", "output": str(output)})
        return read_json(output / "summary.json")
    metadata_dir = new_continuation_metadata(run_root)
    CONTINUATION_LAUNCH_METADATA = metadata_dir
    (metadata_dir / "source_snapshot.zip").write_bytes(payload)
    # Keep the original environment/config/source snapshot and failed batch.
    environment = ensure_runtime(checkout, metadata_dir)
    command = [sys.executable, "-u", "-m", "experiments.run_channel_precompensation_v1",
               "--resume-from", str(original), "--execute", "--output", str(output)]
    log_path = metadata_dir / "runner.log"
    try:
        with log_path.open("w", encoding="utf-8") as runner_log:
            with subprocess.Popen(command, cwd=checkout, env=environment,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                  text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    runner_log.write(line)
                    runner_log.flush()
                returncode = process.wait()
        write_json(metadata_dir / "runner_process.json",
                   {"returncode": returncode, "log_path": str(log_path)})
    except BaseException as error:
        write_json(metadata_dir / "launch_error.json",
                   {"error": type(error).__name__ + ": " + str(error), "log_path": str(log_path)})
        raise
    try:
        summary = read_json(output / "summary.json")
    except (OSError, ValueError) as error:
        raise RuntimeError("Continuation produced no readable summary; runner return code "
                           + str(returncode) + "; log: " + str(log_path)
                           + "; original evidence is retained.") from error
    print({"runner_exit": returncode, "runner_log": str(log_path),
           "method_summaries": summary.get("method_summaries", {}),
           "note": "Only saved zero observations are reused; failed or unreachable writes remain in the denominator."})
    return summary


def package_continuation(config, run_root):
    run_root = Path(run_root)
    summary_path = run_root / CONTINUATION_NAME / "summary.json"
    try:
        summary = read_json(summary_path) if summary_path.is_file() else {}
    except (OSError, ValueError) as error:
        summary = {"read_error": str(error)}
    original = read_json(run_root / "batch/summary.json")
    terminal = terminal_counts(summary)
    rows = {row["source_id"]: row for row in summary.get("sources", [])}
    report = {"scope": "layout_fix_continuation_of_saved_zero_batch", "config": config,
              "counts_terminal": terminal,
              "prior_actual_calls": original.get("actual_calls", {}),
              "actual_calls": summary.get("actual_calls") if terminal else None,
              "last_recorded_calls": summary.get("actual_calls", {}),
              "cumulative_actual_calls": summary.get("cumulative_actual_calls") if terminal else None,
              "reused_counts": summary.get("reused_counts", {}),
              "method_summaries": summary.get("method_summaries", {}),
              "sources": [rows.get(row["source_id"], {"source_id": row["source_id"], "status": "not_observed"})
                          for row in config["sources"]],
              "claim": "Engineering continuation; original failure retained; no scientific PASS."}
    write_json(run_root / (CONTINUATION_NAME + "_summary.json"), report)
    destination = run_root.parent / (run_root.name + "-continuation-layout-v2-results.zip")
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run_root.rglob("*")):
            if path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(run_root.resolve()):
                archive.write(path, Path(run_root.name) / path.relative_to(run_root))
    return report, destination


In [ ]:
#@title 本次源码快照（无需上传代码包）
SOURCE_BUNDLE_B64 = (
    '...'
    'AGbgAAAAAA=='
)


In [ ]:
SOURCE_PAYLOAD = base64.b64decode(SOURCE_BUNDLE_B64)
CHECKOUT = unpack_source(SOURCE_PAYLOAD, Path(tempfile.mkdtemp(prefix="cegwm-precomp-continue-")) / "source")
CONFIG = read_json(RESUME_BATCH / "config.json")
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print({"resume_from": str(RESUME_BATCH), "new_output": str(RUN_ROOT / CONTINUATION_NAME),
       "quality": "report_only", "regenerate_zero_images": False})


## 执行剩余步骤

原零臂图像、配置和失败文件保持原样。新梯度及失败前诊断逐步保存。若仍有前向不一致，会保留证据并停止相应来源，不会放宽门槛、调参、换来源或隐式重试。


In [ ]:
try:
    SUMMARY = run_or_show_continuation(CHECKOUT, RUN_ROOT, SOURCE_PAYLOAD)
except Exception as error:
    metadata_dir = CONTINUATION_LAUNCH_METADATA or new_continuation_metadata(RUN_ROOT)
    write_json(metadata_dir / "notebook_error.json", {"error": type(error).__name__ + ": " + str(error)})
    print({"execution_stopped": type(error).__name__ + ": " + str(error), "original_retained": str(RESUME_BATCH)})
finally:
    AGGREGATE, RESULTS_ZIP = package_continuation(CONFIG, RUN_ROOT)


## 汇总与回传

本格仅打包现有文件，包含原失败批次和本次续跑。无需重新运行上面的模型格；回传下载的 ZIP 即可。


In [ ]:
AGGREGATE, RESULTS_ZIP = package_continuation(CONFIG, RUN_ROOT)
print(json.dumps({"counts_terminal": AGGREGATE["counts_terminal"],
                  "prior_actual_calls": AGGREGATE["prior_actual_calls"],
                  "actual_calls": AGGREGATE["actual_calls"],
                  "cumulative_actual_calls": AGGREGATE["cumulative_actual_calls"],
                  "reused_counts": AGGREGATE["reused_counts"],
                  "method_summaries": AGGREGATE["method_summaries"],
                  "sources": [{"source_id": row["source_id"], "status": row.get("status")}
                              for row in AGGREGATE["sources"]],
                  "results_zip": str(RESULTS_ZIP)}, indent=2, ensure_ascii=False))
from google.colab import files
files.download(str(RESULTS_ZIP))
